# 09 - Cost, observability, and putting an agent in production

An agent that works on your laptop and an agent you can leave running are
different artefacts. This notebook is the difference.

**Docs:** [Track cost and usage](https://code.claude.com/docs/en/agent-sdk/cost-tracking) ·
[Hosting the Agent SDK](https://code.claude.com/docs/en/agent-sdk/hosting)

In [1]:
# --- boot cell: run this first in every notebook -------------------------
import os, sys, json, asyncio
from pathlib import Path
from dotenv import load_dotenv

# The SDK reads ANTHROPIC_API_KEY from the process environment.
# It does NOT parse .env for you, so we do it here.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Copy .env.example to .env and add your key."
)

MODEL = os.environ.get("COURSE_MODEL") or None   # None = SDK default
print("key loaded, model =", MODEL or "(SDK default)")

key loaded, model = claude-haiku-5-5


In [2]:
from claude_agent_sdk import (
    query, ClaudeAgentOptions, AssistantMessage, ResultMessage,
)

SCRATCH = ROOT / "workspace" / "_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)
(SCRATCH / "orders.csv").write_text(
    "id,customer,total\nA-1001,Rao,82.50\nA-1002,Webb,19.99\n", encoding="utf-8"
)
print("ready")

ready


## 9.1 Where the numbers live

| Field | On | Scope |
|---|---|---|
| `total_cost_usd` | `ResultMessage` | whole call, **subagents included** |
| `usage` | `ResultMessage` | **main loop only** — undercounts once you have subagents |
| `model_usage` | `ResultMessage` | whole tree, broken down by model |
| `num_turns` | `ResultMessage` | tool-use round trips |
| `usage`, `message_id` | `AssistantMessage` | per step |

> **These are client-side estimates**, computed locally from a bundled price
> table. They drift when pricing changes or the SDK does not recognise a model.
> Use them for development insight and budgeting. For authoritative billing, use
> the [Usage and Cost API](https://platform.claude.com/docs/en/build-with-claude/usage-cost-api)
> or the Console. Never bill an end user from `total_cost_usd`.

Three traps worth memorising:

1. **`usage` excludes subagents.** Use `model_usage` or `total_cost_usd`.
2. **Per-step `output_tokens` is a placeholder.** Only the result message
   carries the real output count.
3. **Parallel tool calls share a `message_id`.** Deduplicate by ID or you double-count.

In [3]:
seen, input_tokens = set(), 0
result = None

try:
    async for m in query(
        prompt="Read orders.csv and give me the total revenue.",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob"], tools=["Read", "Glob"],
            permission_mode="dontAsk", cwd=str(SCRATCH),
            setting_sources=[], max_turns=8, model=MODEL,
        ),
    ):
        if isinstance(m, AssistantMessage) and not m.parent_tool_use_id:
            mid = m.message_id
            if mid and mid not in seen:            # dedupe parallel tool calls
                seen.add(mid)
                usage = m.usage or {}
                input_tokens += usage.get("input_tokens", 0)
        elif isinstance(m, ResultMessage):
            result = m
except Exception as exc:
    print("raised:", str(exc)[:80])

print(f"steps (unique message ids) : {len(seen)}")
print(f"input tokens (summed)      : {input_tokens}")
if result:
    print(f"turns                      : {result.num_turns}")
    print(f"result.usage (main loop)   : {result.usage}")
    print(f"total_cost_usd             : ${result.total_cost_usd or 0:.5f}")
    print(f"model_usage (whole tree)   : {result.model_usage}")

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


steps (unique message ids) : 3
input tokens (summed)      : 6
turns                      : 3
result.usage (main loop)   : {'input_tokens': 6, 'cache_creation_input_tokens': 1996, 'cache_read_input_tokens': 3595, 'output_tokens': 297, 'output_tokens_details': {'thinking_tokens': 137}, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 1996}, 'inference_geo': 'global', 'iterations': [{'input_tokens': 2, 'output_tokens': 126, 'cache_read_input_tokens': 1874, 'cache_creation_input_tokens': 122, 'cache_creation': {'ephemeral_5m_input_tokens': 122, 'ephemeral_1h_input_tokens': 0}, 'type': 'message'}], 'speed': 'standard'}
total_cost_usd             : $0.00140
model_usage (whole tree)   : {'claude-haiku-5-5': {'inputTokens': 6, 'outputTokens': 297, 'cacheReadInputTokens': 3595, 'cacheCreationInputTokens': 1996, 'webSearchRequests': 0, 'costUSD': 0.00043455, 'context

### Cache tokens

The SDK uses prompt caching automatically — you configure nothing. Two extra
fields tell you whether it is working:

- `cache_creation_input_tokens` — writing new cache entries (charged higher)
- `cache_read_input_tokens` — reading existing entries (charged lower)

A healthy long session shows cache **reads** dominating after the first turn.
If they stay near zero, something changes on every request — a timestamp in the
system prompt, a per-run ID, dynamically ordered tool definitions — and you are
paying full input price every turn.

For short repeated sessions against the same system prompt with gaps over five
minutes, set `ENABLE_PROMPT_CACHING_1H` to request a one-hour TTL.

In [ ]:
if result and result.usage:
    u = result.usage
    print(f"input            : {u.get('input_tokens', 0)}")
    print(f"output           : {u.get('output_tokens', 0)}")
    print(f"cache creation   : {u.get('cache_creation_input_tokens', 0)}")
    print(f"cache read       : {u.get('cache_read_input_tokens', 0)}")

## 9.2 Accumulating across calls

The SDK gives you **no session-level total**. Each `query()` reports only its own
call. Keep the running total yourself — and count error results too, because a
run that failed still spent money.

In [4]:
async def run_and_bill(prompt, ledger):
    """Run one query; always record its cost, success or failure."""
    try:
        async for m in query(
            prompt=prompt,
            options=ClaudeAgentOptions(
                allowed_tools=["Read", "Glob"], tools=["Read", "Glob"],
                permission_mode="dontAsk", cwd=str(SCRATCH),
                setting_sources=[], max_turns=6, model=MODEL,
            ),
        ):
            if isinstance(m, ResultMessage):
                cost = m.total_cost_usd or 0.0
                ledger.append({"subtype": m.subtype, "cost": cost,
                               "turns": m.num_turns, "session": m.session_id})
                print(f"  {m.subtype:<24} ${cost:.5f}")
    except Exception as exc:
        # The result branch above has usually already recorded the cost.
        print(f"  raised: {str(exc)[:60]}")


ledger = []
for p in ["How many orders are in orders.csv?", "What is the largest order total?"]:
    await run_and_bill(p, ledger)

print(f"\ncalls: {len(ledger)}   total: ${sum(e['cost'] for e in ledger):.5f}")

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


  success                  $0.00134


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors
[claude-code:unrecognized_model] {"model":"claude-haiku-5-5","query_source":"sdk"}


  success                  $0.00118

calls: 2   total: $0.00252


### Streaming input mode is different

In [streaming input mode](https://code.claude.com/docs/en/agent-sdk/streaming-vs-single-mode),
one `query()` call carries many user turns and **each turn emits its own
result**:

- `usage` — that turn only, main loop only
- `total_cost_usd` and `model_usage` — running total for the whole call

So read the **latest** result for call totals; do not sum across results. The
running totals reset when your app sends `/clear`, `/reset`, or `/new` — to
total a call with resets, add the last result before each reset to the final
one.

## 9.3 Handling every ending

Production code has to handle all five result subtypes, not just `success`.

In [ ]:
def handle(result: ResultMessage) -> dict:
    """The branch table every production agent needs."""
    base = {
        "session_id": result.session_id,
        "cost": result.total_cost_usd or 0.0,
        "turns": result.num_turns,
        "stop_reason": result.stop_reason,
    }

    if result.subtype == "success":
        # stop_reason == "refusal" means the model declined the request.
        if result.stop_reason == "refusal":
            return {**base, "action": "flag_for_human", "output": None}
        return {**base, "action": "accept", "output": result.result}

    if result.subtype == "error_max_turns":
        # The work so far is preserved. Resume with a higher cap.
        return {**base, "action": "resume", "output": None}

    if result.subtype == "error_max_budget_usd":
        # Do not auto-retry a budget failure - that is how you spend twice.
        return {**base, "action": "alert_and_stop", "output": None}

    if result.subtype == "error_during_execution":
        # After a crash the cost fields may be ZEROED. Recover totals from the
        # last good result, or by summing per-step usage.
        return {**base, "action": "retry_once_then_alert", "output": None}

    return {**base, "action": "alert_and_stop", "output": None}


try:
    async for m in query(
        prompt="Read orders.csv, then write a detailed 500-word analysis to report.md.",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Write", "Glob"], tools=["Read", "Write", "Glob"],
            permission_mode="acceptEdits", cwd=str(SCRATCH),
            setting_sources=[], max_turns=1,        # deliberately too low
            model=MODEL,
        ),
    ):
        if isinstance(m, ResultMessage):
            print(handle(m))
except Exception as exc:
    print("raised (expected):", str(exc)[:70])

## 9.4 Six levers on cost

In rough order of how much they move the number:

1. **`effort`.** `"low"` for lookups and listings. This is the biggest dial and
   the least used.
2. **Model per agent.** Put cheap agents on a small model with `AgentDefinition.model`.
   In our triage team, set `COURSE_FAST_MODEL=haiku` and the two research agents
   drop; the reviewer keeps the main model.
3. **Fewer tools.** Every schema in context is paid for on every turn.
4. **Subagents for read-heavy subtasks.** The parent's context grows by a
   summary rather than by everything the subagent read.
5. **Summarise in tool handlers.** Returning 4,000 lines of JSON is a cost
   decision, not a formatting one.
6. **Cache stability.** Keep the system prompt and tool order byte-identical
   across runs so the cache actually hits.

## 9.5 Observability

You have three places to instrument, and they answer different questions.

**Hooks — what the agent did.** A `PreToolUse` hook with no matcher gives you
every tool call from every agent in the tree, tagged with `agent_type`, for zero
tokens. This is your audit log; make it the first thing you add.

**The message stream — how it got there.** `AssistantMessage` text blocks are
Claude's reasoning per turn. Log them at debug level; they are how you diagnose
a bad decision after the fact.

**`ResultMessage` — what it cost and how it ended.** Emit one metric line per
run: subtype, turns, cost, session id. Alert on the rate of non-`success`
subtypes, not on individual failures.

A minimal, genuinely useful set of metrics:

| Metric | Alert when |
|---|---|
| `agent_runs_total{subtype}` | non-success rate rises |
| `agent_cost_usd` (p50, p95) | p95 approaches your budget cap |
| `agent_turns` (p95) | p95 approaches `max_turns` |
| `tool_calls_total{tool, agent}` | a tool starts failing or looping |
| `hook_denials_total{reason}` | any denial in a path you thought was clean |

Keep the `session_id` on every log line. It is the join key between your logs
and the transcript on disk — and the thing you need to resume.

In [ ]:
# A structured run record you would emit to your logging pipeline.
import json
from datetime import datetime, timezone
from claude_agent_sdk import HookMatcher

TOOL_CALLS = []

async def observe(input_data, tool_use_id, context):
    TOOL_CALLS.append({
        "tool": input_data.get("tool_name"),
        "agent": input_data.get("agent_type") or "main",
    })
    return {}

TOOL_CALLS.clear()
record = None

try:
    async for m in query(
        prompt="How many orders are in orders.csv?",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob"], tools=["Read", "Glob"],
            permission_mode="dontAsk", cwd=str(SCRATCH),
            setting_sources=[], max_turns=6, model=MODEL,
            hooks={"PreToolUse": [HookMatcher(hooks=[observe])]},
        ),
    ):
        if isinstance(m, ResultMessage):
            record = {
                "ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "session_id": m.session_id,
                "subtype": m.subtype,
                "stop_reason": m.stop_reason,
                "turns": m.num_turns,
                "cost_usd": round(m.total_cost_usd or 0.0, 6),
                "duration_ms": m.duration_ms,
                "tool_calls": TOOL_CALLS,
            }
except Exception as exc:
    print("raised:", str(exc)[:60])

print(json.dumps(record, indent=2))

## 9.6 Deployment notes

**Authentication.** `ANTHROPIC_API_KEY` in the process environment, from your
secret manager. The SDK does not read `.env`. Bedrock, Google Cloud's Agent
Platform, Microsoft Foundry, and Claude Platform on AWS are selected by
environment variable.

> Anthropic does not permit third-party products to offer claude.ai login or
> subscription rate limits for agents built on the SDK. Use API key auth.

**Containers.** Both SDKs bundle a native Claude Code binary, so a plain
`pip install` usually needs nothing extra. Two exceptions: a
source-distribution install (for example ARM64 Windows) has no bundled binary,
and `npm ci --omit=optional` skips it on the TypeScript side. Install Claude
Code natively in those cases.

**Filesystem.** Sessions are written to `~/.claude/projects/...`. In an
ephemeral container that disappears. Either mount a volume, use a
[`session_store` adapter](https://code.claude.com/docs/en/agent-sdk/session-storage),
or design for stateless runs and pass what matters in the prompt.

**`bypassPermissions`** is for isolated environments — CI, a locked-down
container — and nowhere else. It cannot be used as root on Unix.

**Concurrency.** Each `query()` spawns a subprocess. Bound how many run at once
and give each a `max_budget_usd`.

**Docs:** [Hosting the Agent SDK](https://code.claude.com/docs/en/agent-sdk/hosting)

## 9.7 Pre-flight checklist

Before an agent runs unattended, all of these:

**Bounds**
- [ ] `max_turns` set
- [ ] `max_budget_usd` set (the only cap that bounds a subagent tree)
- [ ] Subagent depth and concurrency capped if `Agent` is allowed

**Surface**
- [ ] `tools=[...]` limits availability to what is actually needed
- [ ] `allowed_tools` lists exactly what may run unprompted
- [ ] `permission_mode` is `dontAsk` (headless) — never `bypassPermissions`
      outside an isolated environment
- [ ] Destructive operations are denied by rule **and** by a `PreToolUse` hook

**Correctness**
- [ ] Every result `subtype` has a branch; `result` is read only on `success`
- [ ] `stop_reason == "refusal"` is handled
- [ ] The single-shot `query()` raise is caught
- [ ] Tool handlers validate inputs and return `is_error` with instructions
- [ ] `setting_sources=[]` unless you deliberately want ambient config

**Operations**
- [ ] A `PreToolUse` audit hook writes every call somewhere durable
- [ ] `session_id` is logged on every line and stored with the work item
- [ ] Cost per run is a metric, alerted on p95
- [ ] Secrets come from the environment, never from a prompt or a repo

**Evaluation**
- [ ] A fixed set of cases with expected outcomes exists
- [ ] It runs on every prompt change — prompts are code, and they regress

## 9.8 Exercises

1. Run the multi-agent triage example on all eight tickets. Compute mean and p95
   cost. Which ticket is the outlier, and why?
2. Add `effort="low"` to the two research subagents. Re-run. How much cost went
   away, and did any decision change?
3. Write a `Stop` hook that appends the run record from 9.5 to a JSONL file.
   Then load it with pandas and plot cost per run.
4. Set `max_budget_usd` to just below a ticket's typical cost. Confirm the
   subtype and confirm `total_cost_usd` can exceed the cap slightly.
5. Build the eval harness from the checklist: run all eight tickets, compare
   `priority` and `queue` against a hand-written expected answer, and report
   accuracy. Now change one line of the system prompt and re-run it.

In [ ]:
# Scratch space for the exercises.

## Next

You now have every piece. The last two notebooks put them together on the
support-triage problem and then take the result apart.

[**10 - Build 1: the single-agent system**](10_build_support_triage.ipynb) —
inspect the real project, exercise its tools with no API calls, run the queue,
score it against the routing policy, then break it three ways on purpose.

[**11 - Build 2: the multi-agent team**](11_build_multi_agent_triage.ipynb) —
capability boundaries, four specialists, a guardrail you can fire offline, and
the head-to-head that decides when five agents are worth it.

Both notebooks are self-contained: you build the tools, the agents, and the
scorer in cells, so nothing is hidden behind an import.

---

### References

- [Track cost and usage](https://code.claude.com/docs/en/agent-sdk/cost-tracking)
- [Hosting the Agent SDK](https://code.claude.com/docs/en/agent-sdk/hosting)
- [Session storage](https://code.claude.com/docs/en/agent-sdk/session-storage)
- [Usage and Cost API](https://platform.claude.com/docs/en/build-with-claude/usage-cost-api)
- [Environment variables](https://code.claude.com/docs/en/env-vars)